# Signified — make a graph JSON in Colab

This notebook is the measurement step. It does **not** interpret the graph.

1. Runtime → Change runtime type → **T4 GPU**
2. Accept the Gemma licence: https://huggingface.co/google/gemma-2-2b
3. Run the cells in order
4. Download the JSON and put it in Signified's `data/` folder

First graph to generate (matches the public circuit-tracer demo):

`Fact: The capital of the state containing Dallas is` → Austin

To generate the Signified canonical prompt later, change `PROMPT` to `The capital of Australia is`.

In [ ]:
# @title Install circuit-tracer and log in to Hugging Face
try:
    import google.colab  # noqa: F401

    !mkdir -p repository && cd repository && \
      git clone --depth 1 https://github.com/safety-research/circuit-tracer && \
      curl -LsSf https://astral.sh/uv/install.sh | sh && \
      uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not Colab — install circuit-tracer in your own venv first.")

In [ ]:
from pathlib import Path

import torch
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files

PROMPT = "Fact: The capital of the state containing Dallas is"
SLUG = "dallas-austin"

model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b",
    "gemma",
    dtype=torch.bfloat16,
    backend="transformerlens",
)

In [ ]:
graph = attribute(
    prompt=PROMPT,
    model=model,
    max_n_logits=10,
    desired_logit_prob=0.95,
    batch_size=256,
    max_feature_nodes=5000,
    offload="disk" if IN_COLAB else "cpu",
    verbose=True,
)

graph_dir = Path("graphs")
graph_dir.mkdir(exist_ok=True)
graph_path = graph_dir / f"{SLUG}.pt"
graph.to_pt(graph_path)
print("saved raw graph", graph_path, "— large; do not send this to Signified")

In [ ]:
graph_file_dir = "./graph_files"
create_graph_files(
    graph_or_path=graph_path,
    slug=SLUG,
    output_path=graph_file_dir,
    node_threshold=0.8,
    edge_threshold=0.98,
)

json_path = Path(graph_file_dir) / f"{SLUG}.json"
print("Signified wants this file:", json_path, json_path.stat().st_size, "bytes")

if IN_COLAB:
    from google.colab import files

    files.download(str(json_path))